In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/var/home/alexis/src/llm-zoomcamp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10379.70it/s]


In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.32332402)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.019730462)

In [8]:
from ingest import load_faq_data

documents = load_faq_data()

In [9]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [10]:
from tqdm.auto import tqdm

In [11]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

100%|██████████| 27/27 [00:03<00:00,  8.82it/s]


1350

In [12]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [13]:
import numpy as np
X = np.array(vectors)

In [14]:
scores = X.dot(v_query)

In [15]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [16]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [17]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [18]:
top5 = np.argsort(scores)[-5:]

In [19]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [20]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.71921325
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

Vector Search with minsearch

In [21]:
#Creating the index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [22]:
#Searching

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [23]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [24]:
#Filtering by course
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

RAG with Vector Search

In [25]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [26]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [27]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index, 
    llm_client=openai_client
)

In [28]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [29]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [30]:
vector_assistant = RAGVector(
    embedder=model, 
    index=vindex, 
    llm_client=openai_client
)

In [31]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. Start learning and submit your work while the submission form is open. If you want a certificate, make sure to submit your project before submissions close.'

Vector Search with sqlitesearch

In [32]:
# Creating the index
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'], 
    mode='ivf', 
    db_path='faq_vectors_2.db'
)

In [33]:
#Indexing the data
vs_index.fit(vectors, documents)

In [34]:
# Searching
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [35]:
# Closing the connection
vs_index.close()

Reopening the index

In a new Python session, you can reopen the index without re-computing embeddings.

Continue in notebook_2